In [1]:
import os
import sys
import torch
import numpy as np

# Ensure access to src/
sys.path.append(os.path.abspath('..'))
from src.cnn_encoder import StandaloneCNNEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing Phase 6 unit tests on device: {device}")

# -------------------------------------------------------------
# Test Case 1: LCL Pretraining Window Shape (Phase 4 Artifact)
# Shape: (Batch=16, Length=48, Channels=25)
# -------------------------------------------------------------
lcl_batch = torch.randn(16, 48, 25).to(device)

cnn_lcl = StandaloneCNNEncoder(
    in_channels=25,
    cnn_hidden_dims=(64, 128),
    kernel_size=3,
    pool_type="max",
    pool_size=2
).to(device)

out_lcl = cnn_lcl(lcl_batch)
print("\n--- Test Case 1: LCL Pretraining Tensor Verification ---")
print(f"Input Shape:  {tuple(lcl_batch.shape)}")
print(f"Output Shape: {tuple(out_lcl.shape)}  (Expected: 16, 24, 128)")
assert out_lcl.shape == (16, 24, 128), "Mismatch in LCL output shape!"

# -------------------------------------------------------------
# Test Case 2: Feeder Downstream FDI Window Shape (Phase 5 Dataset)
# Shape: (Batch=32, Length=30, Channels=12)
# -------------------------------------------------------------
fdi_batch = torch.randn(32, 30, 12).to(device)

# Preserving temporal dimension for BiLSTM downstream ingestion (pool_type=None)
cnn_fdi_preserved = StandaloneCNNEncoder(
    in_channels=12,
    cnn_hidden_dims=(32, 64),
    kernel_size=3,
    pool_type=None
).to(device)

out_fdi_preserved = cnn_fdi_preserved(fdi_batch)
print("\n--- Test Case 2: Feeder Preserved Sequence Verification ---")
print(f"Input Shape:  {tuple(fdi_batch.shape)}")
print(f"Output Shape: {tuple(out_fdi_preserved.shape)}  (Expected: 32, 30, 64)")
assert out_fdi_preserved.shape == (32, 30, 64), "Mismatch in Feeder sequence shape!"

# -------------------------------------------------------------
# Test Case 3: Global Sequence Pooling (Vector Embedding)
# -------------------------------------------------------------
cnn_global = StandaloneCNNEncoder(
    in_channels=12,
    cnn_hidden_dims=(32, 64, 128),
    pool_type="global_avg"
).to(device)

out_global = cnn_global(fdi_batch)
print("\n--- Test Case 3: Global Temporal Pooling Verification ---")
print(f"Input Shape:  {tuple(fdi_batch.shape)}")
print(f"Output Shape: {tuple(out_global.shape)}  (Expected: 32, 128)")
assert out_global.shape == (32, 128), "Mismatch in global pooling shape!"

# -------------------------------------------------------------
# Test Case 4: Gradient Flow & Backward Pass Verification
# -------------------------------------------------------------
loss = out_global.sum()
loss.backward()

grad_norms = [p.grad.norm().item() for p in cnn_global.parameters() if p.grad is not None]
has_non_zero_grads = all(g > 0.0 for g in grad_norms)

print("\n--- Test Case 4: Gradient Flow Check ---")
print(f"All parameters receiving gradients: {has_non_zero_grads}")
print(f"Mean parameter gradient norm:      {np.mean(grad_norms):.6f}")
assert has_non_zero_grads, "Zero gradient detected across parameters!"

print("\n[SUCCESS] Phase 6 CNN Component verified and operational.")

Executing Phase 6 unit tests on device: cuda

--- Test Case 1: LCL Pretraining Tensor Verification ---
Input Shape:  (16, 48, 25)
Output Shape: (16, 24, 128)  (Expected: 16, 24, 128)

--- Test Case 2: Feeder Preserved Sequence Verification ---
Input Shape:  (32, 30, 12)
Output Shape: (32, 30, 64)  (Expected: 32, 30, 64)

--- Test Case 3: Global Temporal Pooling Verification ---
Input Shape:  (32, 30, 12)
Output Shape: (32, 128)  (Expected: 32, 128)

--- Test Case 4: Gradient Flow Check ---
All parameters receiving gradients: True
Mean parameter gradient norm:      56.751483

[SUCCESS] Phase 6 CNN Component verified and operational.
